In [1]:
import pandas as pd
import math
from datetime import datetime as _dt, timedelta as _td
from functools import lru_cache
from IPython.display import display
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    raise ImportError("Debe instalar el paquete seaborn: pip install seaborn")
try:
    import yfinance as yf
except ImportError:
    raise ImportError("Debe instalar el paquete yfinance: pip install yfinance")

In [2]:
# Descarga precios y calcula correlación entre tickers (desde 2015)
# Ejecutá primero la celda 1 (importaciones) si aún no lo hiciste.
if yf is None:
    raise ImportError("yfinance no disponible. Ejecutá la celda de importaciones e instalá yfinance si hace falta.")

try:
    tickers = ["MCHI", "SPY", "EWJ", "EWY", "ARGT", "EWZ", "EWL", "EWA", "EWU", "EWG", "EWC", "INDA"]
    # Mostrar solo la cantidad de tickers para evitar imprimir la lista completa
    print(f"Descargando {len(tickers)} tickers (inicio 2015-01-01). Esto puede tardar...")
    data = yf.download(tickers, start="2015-01-01", progress=False, auto_adjust=False)

    # Robustamente extraer Adjusted Close
    precios = None
    if isinstance(data.columns, pd.MultiIndex):
        lvl0 = list(data.columns.get_level_values(0))
        lvl1 = list(data.columns.get_level_values(1))
        if "Adj Close" in set(lvl0):
            # forma común: nivel 0 = fields, nivel 1 = tickers
            precios = data.xs("Adj Close", axis=1, level=0)
        elif "Adj Close" in set(lvl1):
            # forma alternativa: nivel 0 = tickers, nivel 1 = fields
            precios = data.xs("Adj Close", axis=1, level=1)
        else:
            precios = None
    else:
        # Estructura plana (poca cantidad de tickers) -> buscar columna 'Adj Close' o 'Close'
        if "Adj Close" in data.columns:
            precios = data[["Adj Close"]]
        elif "Close" in data.columns:
            precios = data[["Close"]]
        else:
            precios = None

    # Si no logramos extraer precios, reintentar por ticker individualmente (más costoso)
    if precios is None or (hasattr(precios, 'empty') and precios.empty):
        print("No se detectó 'Adj Close' en la descarga masiva — reintentando descarga por ticker (más lenta)...")
        lista = []
        for t in tickers:
            try:
                td = yf.download(t, start="2015-01-01", progress=False, auto_adjust=False)
                if isinstance(td.columns, pd.MultiIndex):
                    # intentar extraer Adj Close
                    if "Adj Close" in list(td.columns.get_level_values(0)):
                        s = td.xs("Adj Close", axis=1, level=0)
                    elif "Adj Close" in list(td.columns.get_level_values(1)):
                        s = td.xs("Adj Close", axis=1, level=1)
                    else:
                        s = td.get("Adj Close") if "Adj Close" in td.columns else td.get("Close")
                else:
                    s = td.get("Adj Close") if "Adj Close" in td.columns else td.get("Close")
                if s is None or (hasattr(s, 'empty') and s.empty):
                    print(f"Advertencia: no hay serie de precios para {t}")
                    continue
                # asegurar serie con nombre de columna = ticker
                s = s.rename(t)
                lista.append(s)
            except Exception as e:
                print(f"Error descargando {t}: {e}")
                continue
        if lista:
            precios = pd.concat(lista, axis=1)
        else:
            precios = pd.DataFrame()

    # Asegurar DataFrame y columnas legibles
    if isinstance(precios, pd.Series):
        precios = precios.to_frame()
    if not precios.empty:
        precios.columns = [str(c) for c in precios.columns]
    else:
        raise ValueError("No se encontraron series de precios (Adj Close/Close) tras reintentos. Revisa la conexión o los tickers.")

    # Calcular retornos diarios y matriz de correlación
    retornos = precios.pct_change().dropna(how="all")
    retornos = retornos.dropna(axis=1, how="all")
    if retornos.empty:
        raise ValueError("No hay retornos calculables tras el dropna.")

    corr = retornos.corr().round(3)

    # Mostrar matriz con Styler (si el frontend lo soporta)
    try:
        styled = corr.style.background_gradient(cmap="RdYlGn").format("{:.3f}")
        display(styled)
    except Exception:
        print("No se pudo formatear la tabla con Styler; imprimo la matriz raw:")
        print(corr)

    # Se elimina la gráfica y la impresión de la lista de correlaciones por solicitud del usuario.

except Exception as e:
    print(f"Error al calcular correlaciones: {e}")
    raise

Descargando 12 tickers (inicio 2015-01-01). Esto puede tardar...


,ARGT,EWA,EWC,EWG,EWJ,EWL,EWU,EWY,EWZ,INDA,MCHI,SPY
ARGT,1.000,0.564,0.621,0.551,0.523,0.509,0.560,0.530,0.566,0.459,0.450,0.607
EWA,0.564,1.000,0.824,0.760,0.708,0.745,0.821,0.709,0.618,0.642,0.558,0.793
EWC,0.621,0.824,1.000,0.750,0.676,0.723,0.814,0.668,0.621,0.612,0.522,0.803
EWG,0.551,0.760,0.750,1.000,0.723,0.818,0.850,0.685,0.535,0.612,0.551,0.774
EWJ,0.523,0.708,0.676,0.723,1.000,0.674,0.710,0.658,0.500,0.566,0.518,0.743
EWL,0.509,0.745,0.723,0.818,0.674,1.000,0.813,0.628,0.517,0.590,0.509,0.739
EWU,0.560,0.821,0.814,0.850,0.710,0.813,1.000,0.673,0.589,0.641,0.547,0.763
EWY,0.530,0.709,0.668,0.685,0.658,0.628,0.673,1.000,0.553,0.619,0.618,0.689
EWZ,0.566,0.618,0.621,0.535,0.500,0.517,0.589,0.553,1.000,0.548,0.445,0.554
INDA,0.459,0.642,0.612,0.612,0.566,0.590,0.641,0.619,0.548,1.000,0.469,0.633


In [6]:
# Dashboard: correlación vs SPY + P/E de cada ticker (solo TrailingPE)
# Ejecutar esta celda después de la celda de descarga/imports. Puede tardar: hace descargas por ticker si hace falta.
from IPython.display import display
import pandas as pd
import time
# Intentaremos usar ipywidgets para interacción; si no está disponible, usamos una tabla estática
try:
    import ipywidgets as widgets
    has_widgets = True
except Exception:
    has_widgets = False

# Lista de tickers (ajustá según necesites) -- reusa la misma lista usada arriba
tickers = ["MCHI", "SPY", "EWJ", "EWY", "ARGT", "EWZ", "EWL", "EWA", "EWU", "EWG", "EWC", "INDA",
"EWW", "EWQ", "EWI", "EWP", "EWT", "EWH", "EZA", "EWS", "EWD", "TUR", "EIDO", "EWM", "THD",
"EPOL", "ERUS", "EIRL", "EIS", "ENZL", "EPU", "ECH", "EGPT", "GREK", "EFNL", "NORW", "EWD",
"EDEN", "EWN", "EWD", "EZA", "EPI", "EWM", "EIDO", "TUR", "EWW", "EIS", "ENZL", "EPU", "ECH"]
start = '2015-01-01'
print(f'Construyendo dashboard para {len(tickers)} tickers. Esto puede tardar...')
# Reusar la lógica robusta para extraer 'Adj Close' del cell anterior (si ya corriste la celda masiva, el objeto 'data' puede existir).
precios = None
try:
    # Si existe 'data' en el namespace y tiene series suficientes, intentamos extraer adj close desde allí
    if 'data' in globals():
        d = globals().get('data')
        if isinstance(d, pd.DataFrame):
            if isinstance(d.columns, pd.MultiIndex):
                lvl0 = list(d.columns.get_level_values(0))
                lvl1 = list(d.columns.get_level_values(1))
                if 'Adj Close' in set(lvl0):
                    precios = d.xs('Adj Close', axis=1, level=0)
                elif 'Adj Close' in set(lvl1):
                    precios = d.xs('Adj Close', axis=1, level=1)
            else:
                if 'Adj Close' in d.columns:
                    precios = d[['Adj Close']]
                elif 'Close' in d.columns:
                    precios = d[['Close']]
    # Si no hay precios útiles, forzamos nueva descarga masiva
    if precios is None or (hasattr(precios,'empty') and precios.empty):
        import yfinance as yf
        data2 = yf.download(tickers, start=start, progress=False, auto_adjust=False)
        if isinstance(data2.columns, pd.MultiIndex):
            if 'Adj Close' in list(data2.columns.get_level_values(0)):
                precios = data2.xs('Adj Close', axis=1, level=0)
            elif 'Adj Close' in list(data2.columns.get_level_values(1)):
                precios = data2.xs('Adj Close', axis=1, level=1)
        else:
            if 'Adj Close' in data2.columns:
                precios = data2[['Adj Close']]
            elif 'Close' in data2.columns:
                precios = data2[['Close']]
except Exception:
    precios = None

# Fallback por ticker si no encontramos 'Adj Close' de forma masiva
if precios is None or (hasattr(precios,'empty') and precios.empty):
    print('No se detectó Adj Close en la descarga masiva — reintentando por ticker (más lento)...')
    import yfinance as yf
    series_list = []
    for t in tickers:
        try:
            td = yf.download(t, start=start, progress=False, auto_adjust=False)
            s = None
            if isinstance(td.columns, pd.MultiIndex):
                if 'Adj Close' in list(td.columns.get_level_values(0)):
                    s = td.xs('Adj Close', axis=1, level=0)
                elif 'Adj Close' in list(td.columns.get_level_values(1)):
                    s = td.xs('Adj Close', axis=1, level=1)
            else:
                if 'Adj Close' in td.columns:
                    s = td['Adj Close']
                elif 'Close' in td.columns:
                    s = td['Close']
            if s is None or (hasattr(s, 'empty') and s.empty):
                print(f'Advertencia: no hay serie de precios para {t}')
                continue
            # Si s es DataFrame (cuando xs devuelve DF), tomar la columna correcta o renombrar
            if isinstance(s, pd.DataFrame):
                # si la columna ya usa el ticker como nombre, ok; si no, tomamos la primera columna
                if len(s.columns) > 1:
                    s = s.iloc[:, 0]
            s = s.rename(t)
            series_list.append(s)
        except Exception as e:
            print(f'Error descargando {t}: {e}')
            continue
    if series_list:
        precios = pd.concat(series_list, axis=1)
    else:
        raise ValueError('No se obtuvieron series de precios tras los reintentos. Revisa la conexión.')

# Normalizar columnas y nombres
if isinstance(precios, pd.Series):
    precios = precios.to_frame()
precios.columns = [str(c) for c in precios.columns]

# Calcular retornos diarios y matriz de correlación (requerido SPY)
retornos = precios.pct_change().dropna(how='all')
retornos = retornos.dropna(axis=1, how='all')
if retornos.empty:
    raise ValueError('No hay retornos calculables.')
corr = retornos.corr().round(3)
corr_vs_spy = None
if 'SPY' in corr.columns:
    corr_vs_spy = corr['SPY'].rename('Corr_vs_SPY')
else:
    # calcular manualmente correlación con la serie SPY (si existe en precios)
    if 'SPY' in retornos.columns:
        spy_ret = retornos['SPY']
        vals = []
        for col in retornos.columns:
            if col == 'SPY':
                vals.append(1.0)
                continue
            common = pd.concat([spy_ret, retornos[col]], axis=1).dropna()
            if len(common) >= 5:
                vals.append(common.corr().iloc[0,1])
            else:
                vals.append(float('nan'))
        corr_vs_spy = pd.Series(vals, index=retornos.columns, name='Corr_vs_SPY')
    else:
        raise ValueError('No se encuentra SPY para comparar la correlación.')

# Para cada ticker obtener Trailing P/E desde yfinance.info (si está disponible)
import yfinance as yf
pe_list = []
price_list = []
name_list = []
for t in precios.columns:
    try:
        tt = yf.Ticker(t)
        info = tt.info if hasattr(tt, 'info') else {}
        trailing = info.get('trailingPE') if isinstance(info, dict) else None
        # precio actual: preferimos regularMarketPrice desde info, sino último adj close
        p = info.get('regularMarketPrice') if isinstance(info, dict) else None
        if p is None:
            try:
                p = precios[t].dropna().iloc[-1]
            except Exception:
                p = None
        nm = info.get('shortName') or info.get('longName') if isinstance(info, dict) else None
    except Exception as e:
        trailing = None; p = None; nm = None
    pe_list.append(trailing)
    price_list.append(p)
    name_list.append(nm)
    # evitar peticiones rápidas consecutivas que puedan ser bloqueadas
    time.sleep(0.1)

# Construir DataFrame final (sin ForwardPE)
df = pd.DataFrame({
    'Ticker': list(precios.columns),
    'Name': name_list,
    'Price': price_list,
    'TrailingPE': pe_list,
    'Corr_vs_SPY': [corr_vs_spy.get(t, float('nan')) if corr_vs_spy is not None else float('nan') for t in precios.columns]
})
df = df.set_index('Ticker')
# Limpieza: convertir None a NaN y forzar columnas numéricas donde aplique
import numpy as np
df = df.replace({None: np.nan})
for col in ['Price', 'TrailingPE', 'Corr_vs_SPY']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Mostrar tabla: interactive widgets (ordenar, asc/desc, top N) si están disponibles
formats = {'Price': '{:.2f}', 'TrailingPE': '{:.2f}', 'Corr_vs_SPY': '{:.3f}'}
if has_widgets:
    sort_by = widgets.Dropdown(options=df.columns.tolist(), value='Corr_vs_SPY', description='Ordenar por:')
    desc_toggle = widgets.ToggleButton(value=True, description='Descendente')
    top_n = widgets.IntSlider(min=1, max=max(1, len(df)), value=min(10, len(df)), description='Top N')
    out = widgets.Output()

    def render_table(_=None):
        with out:
            out.clear_output()
            ascending = not desc_toggle.value
            to_show = df.sort_values(by=sort_by.value, ascending=ascending).head(top_n.value)
            try:
                display(to_show.style.format(formats).background_gradient(cmap='RdYlGn', subset=['Corr_vs_SPY']))
            except Exception:
                display(to_show)
    sort_by.observe(render_table, names='value')
    desc_toggle.observe(render_table, names='value')
    top_n.observe(render_table, names='value')
    display(widgets.HBox([sort_by, desc_toggle, top_n]))
    display(out)
    # render once
    render_table()
else:
    print('ipywidgets no disponible; mostrando tabla estática ordenada por Corr_vs_SPY (desc):')
    display(df.sort_values('Corr_vs_SPY', ascending=False).style.format(formats).background_gradient(cmap='RdYlGn', subset=['Corr_vs_SPY']))

# Nota: Exportación CSV deshabilitada por solicitud del usuario (no se crea ningún archivo aquí)


Construyendo dashboard para 50 tickers. Esto puede tardar...


Output()

### ¿Qué son Trailing PE

- Trailing PE (P/E histórico): el "Trailing Price-to-Earnings" se calcula como el precio actual dividido por las utilidades por acción de los últimos 12 meses (TTM, "trailing twelve months"). Muestra cuánto están pagando los inversores por cada unidad de beneficio ya realizada. Es un dato histórico respaldado por resultados reportados.



MCHI – iShares MSCI China ETF: Este ETF proporciona acceso al mercado accionario de China, enfocándose en empresas chinas de gran y mediana capitalización disponibles a inversores internacionales
blackrock.com
. Posee 555 activos en cartera (a fecha 10 de noviembre de 2025) y cobra una comisión anual (expense ratio) de 0,59%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

SPY – SPDR S&P 500 ETF Trust: ETF que replica el índice S&P 500 de EE.UU., ofreciendo exposición diversificada a unas 500 empresas líderes de gran capitalización en los 11 sectores principales del mercado estadounidense
ssga.com
. Incluye 503 participaciones (al 30 de septiembre de 2025) y tiene un gasto anual de aproximadamente 0,09%
ssga.com
ssga.com
. Fuente oficial: State Street Global Advisors (SPDR).

EWJ – iShares MSCI Japan ETF: Ofrece acceso al mercado accionario de Japón, invirtiendo en compañías japonesas de gran y mediana capitalización
blackrock.com
. Contiene 180 valores en cartera (al 11 de noviembre de 2025) y su ratio de gastos anual es 0,50%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWY – iShares MSCI South Korea ETF: Brinda exposición dirigida a Corea del Sur, cubriendo empresas surcoreanas grandes y medianas
blackrock.com
. Posee 82 participaciones (al 10 de noviembre de 2025) y aplica una comisión anual de 0,59%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

ARGT – Global X MSCI Argentina ETF: ETF enfocado en Argentina, invierte en algunas de las mayores y más líquidas empresas con exposición al mercado argentino
globalxetfs.com
. Mantiene 24 activos en cartera (al 30 de septiembre de 2025) y cobra un gasto anual de 0,59%
assets.globalxetfs.com
. Fuente oficial: Global X ETFs.

EWZ – iShares MSCI Brazil ETF: Ofrece acceso al mercado de Brasil, siguiendo un índice de acciones brasileñas de gran y mediana capitalización
blackrock.com
. Incluye 44 componentes (al 11 de noviembre de 2025) y tiene una comisión anual de 0,59%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWL – iShares MSCI Switzerland ETF: Cubre el mercado de Suiza, con exposición a empresas suizas importantes (grandes y medianas)
blackrock.com
. Contiene 43 holdings en cartera (al 11 de noviembre de 2025) y su expense ratio es 0,50%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWA – iShares MSCI Australia ETF: Replica un índice de Australia, dando acceso al mercado bursátil australiano a través de compañías de gran y mediana capitalización
blackrock.com
. Posee 47 posiciones (al 10 de noviembre de 2025) con un gasto anual de 0,50%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWU – iShares MSCI United Kingdom ETF: Sigue un índice de acciones del Reino Unido, brindando exposición a empresas británicas de gran y mediano tamaño
blackrock.com
. Incluye 74 participaciones (al 10 de noviembre de 2025) y cobra una comisión anual de 0,50%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWG – iShares MSCI Germany ETF: Ofrece inversión en el mercado de Alemania, rastreando un índice compuesto por acciones alemanas
blackrock.com
. Contiene 54 valores en cartera (al 11 de noviembre de 2025) y aplica un expense ratio de 0,50%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWC – iShares MSCI Canada ETF: Proporciona exposición al mercado de Canadá mediante un índice de renta variable canadiense diversificado. Posee entre 90 y 100 holdings típicamente (por ejemplo, 94 acciones al último informe) y su comisión anual es 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

INDA – iShares MSCI India ETF: Acceso amplio a la renta variable de India, replicando un índice de acciones indias de gran y mediana capitalización
blackrock.com
. Contiene 160 valores (al 10 de noviembre de 2025) y tiene un gasto anual de 0,62%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWW – iShares MSCI Mexico ETF: Enfocado en México, sigue un índice de acciones mexicanas líderes. Incluye alrededor de 60 empresas (ej. 59 activos recientemente) y presenta un expense ratio de 0,50% (según el prospecto)
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWQ – iShares MSCI France ETF: Cubre el mercado de Francia, dando acceso a empresas francesas de gran capitalización. Tiene aproximadamente 70 componentes en cartera y un ratio de gastos de 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWI – iShares MSCI Italy ETF: Ofrece exposición a Italia, invirtiendo en las principales acciones italianas. Suele mantener alrededor de 25–30 activos y cobra una comisión anual de 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWP – iShares MSCI Spain ETF: Centrado en España, sigue un índice de acciones españolas. Contiene cerca de 25 valores y su gasto anual es 0,50% (según datos del proveedor)
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWT – iShares MSCI Taiwan ETF: Proporciona acceso a Taiwán, invirtiendo en empresas taiwanesas líderes. Incluye alrededor de 90 participaciones y tiene un expense ratio de 0,59% (mercado emergente)
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWH – iShares MSCI Hong Kong ETF: Cubre el mercado de Hong Kong, con exposición a acciones hongkonesas de gran capitalización. Posee unos 40–50 componentes y aplica un expense ratio de 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EZA – iShares MSCI South Africa ETF: Enfocado en Sudáfrica, rastrea un índice de renta variable sudafricana
blackrock.com
. Mantiene 27 posiciones (al 10 de noviembre de 2025) y su comisión anual es 0,59%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EWS – iShares MSCI Singapore ETF: Da acceso a Singapur, invirtiendo en las principales compañías singapurenses. Contiene cerca de 30 holdings y presenta un expense ratio de 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWD – iShares MSCI Sweden ETF: Ofrece exposición a Suecia, siguiendo un índice de acciones suecas. Cuenta con aproximadamente 30–35 valores en cartera y su gasto anual es 0,54%
blackrock.com
. Fuente oficial: BlackRock (iShares).

TUR – iShares MSCI Turkey ETF: Centrado en Turquía, invierte en un amplio conjunto de acciones turcas
blackrock.com
. Posee 83 activos (al 10 de noviembre de 2025) y tiene un expense ratio de 0,59%
blackrock.com
blackrock.com
. Fuente oficial: BlackRock (iShares).

EIDO – iShares MSCI Indonesia ETF: Cubre Indonesia, proporcionando acceso a las mayores empresas indonesias. Incluye alrededor de 85–90 participaciones y su ratio de gastos es 0,59% (mercado emergente)
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EWM – iShares MSCI Malaysia ETF: Ofrece exposición a Malasia a través de un índice de renta variable malasio. Mantiene cerca de 40–45 activos y aplica una comisión anual de 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

THD – iShares MSCI Thailand ETF: Orientado a Tailandia, invierte en las principales acciones tailandesas. Posee unos 125–130 valores en cartera y cobra un gasto anual de 0,59%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EPOL – iShares MSCI Poland ETF: Da acceso al mercado de Polonia, con inversión en acciones polacas de referencia. Contiene alrededor de 30–35 empresas y presenta un expense ratio de 0,58% (aprox.)
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

ERUS – iShares MSCI Russia ETF: (Actualmente suspendido) Históricamente seguía el mercado de Rusia, con ~25 holdings antes de su cierre, y un expense ratio de 0,57%
etfdb.com
. Fuente oficial: BlackRock (iShares).

EIRL – iShares MSCI Ireland ETF: Cobertura de Irlanda, invierte en las principales compañías irlandesas. Incluye unos 25 componentes y su ratio de gastos es 0,50%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EIS – iShares MSCI Israel ETF: Enfocado en Israel, con exposición a acciones israelíes de gran tamaño. Mantiene ~100 títulos y aplica un expense ratio de 0,59%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

ENZL – iShares MSCI New Zealand ETF: Da acceso a Nueva Zelanda, invirtiendo en las principales empresas neozelandesas. Posee cerca de 25–30 activos y tiene un gasto anual de 0,51% (aprox.)
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EPU – iShares MSCI Peru ETF: Cubre Perú, siguiendo un índice de acciones peruanas. Incluye ~25 valores y su expense ratio es 0,57%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

ECH – iShares MSCI Chile ETF: Enfocado en Chile, ofrece exposición a las mayores empresas chilenas. Contiene ~30 holdings y presenta una comisión anual de 0,57%
etfdb.com
etfdb.com
. Fuente oficial: BlackRock (iShares).

EGPT – VanEck Egypt Index ETF: (Liquidado en 2024) Replicaba un índice de Egipto, dando acceso a las principales acciones egipcias. Hacia su cierre mantenía solo 4 posiciones (por liquidación)
etfdb.com
 y tenía un expense ratio de 1,24%
etfdb.com
. Fuente oficial: VanEck.

GREK – Global X MSCI Greece ETF: Centrado en Grecia, invierte en las compañías más grandes y líquidas de ese país
globalxetfs.com
. Posee 29 activos (al 30 de septiembre de 2025) y cobra un gasto anual de 0,57%
assets.globalxetfs.com
. Fuente oficial: Global X ETFs.

EFNL – iShares MSCI Finland ETF: Cubre Finlandia, con exposición a acciones finlandesas líderes. Cuenta con aproximadamente 40–45 componentes y su ratio de gastos es 0,51% (según datos del proveedor). Fuente oficial: BlackRock (iShares).

NORW – Global X MSCI Norway ETF: Ofrece acceso a Noruega, invirtiendo en empresas noruegas de gran tamaño
globalxetfs.com
. Contiene 57 activos (al 30 de septiembre de 2025) y su expense ratio es 0,50%
assets.globalxetfs.com
. Fuente oficial: Global X ETFs.

EDEN – iShares MSCI Denmark ETF: Orientado a Dinamarca, sigue un índice de acciones danesas. Incluye unos 40 títulos y aplica una comisión anual de 0,53% (aprox.). Fuente oficial: BlackRock (iShares).

EWN – iShares MSCI Netherlands ETF: Centrado en Países Bajos, brinda exposición a las principales empresas holandesas. Posee ~50 participaciones y tiene un gasto anual de 0,50% (según prospecto). Fuente oficial: BlackRock (iShares).

EPI – WisdomTree India Earnings ETF: Este ETF de WisdomTree invierte en empresas de India con beneficios positivos, proporcionando una visión de las acciones indias desde una perspectiva fundamental
wisdomtree.com
. Mantiene 540 empresas en cartera (al 30 de septiembre de 2025) y cobra una comisión anual de 0,84%
wisdomtree.com
. Fuente oficial: WisdomTree.

In [4]:
# Indicador "Buffett" por país: Market Cap de empresas listadas / PIB
# Este bloque obtiene el indicador World Bank CM.MKT.LCAP.GD.ZS
# (Market capitalization of listed domestic companies (% of GDP))
# y lo convierte en el "Buffett indicator" (market cap / GDP = % / 100).

import requests
import pandas as pd

# Lista de países y sus códigos ISO2 para la API del World Bank.
country_codes = {
    "China": "CN",
    "Estados Unidos": "US",
    "Japón": "JP",
    "Corea del Sur": "KR",
    "Argentina": "AR",
    "Brasil": "BR",
    "Suiza": "CH",
    "Australia": "AU",
    "Reino Unido": "GB",
    "Alemania": "DE",
    "Canadá": "CA",
    "India": "IN",
    "México": "MX",
    "Francia": "FR",
    "Italia": "IT",
    "España": "ES",
    "Taiwán": None,        # Taiwan no figura en la API del World Bank
    "Hong Kong": "HK",
    "Sudáfrica": "ZA",
    "Singapur": "SG",
    "Suecia": "SE",
    "Turquía": "TR",
    "Indonesia": "ID",
    "Malasia": "MY",
    "Tailandia": "TH",
    "Polonia": "PL",
    "Rusia": "RU",
    "Irlanda": "IE",
    "Israel": "IL",
    "Nueva Zelanda": "NZ",
    "Perú": "PE",
    "Chile": "CL",
    "Egipto": "EG",
    "Grecia": "GR",
    "Finlandia": "FI",
    "Noruega": "NO",
    "Dinamarca": "DK",
    "Países Bajos": "NL"
}

results = []

for country, code in country_codes.items():
    if code is None:
        results.append({"Country": country, "WB_code": None, "Year": None, "MarketCap_pct_GDP": None, "Buffett_indicator": None, "Note": "No disponible en World Bank API"})
        continue

    url = f"http://api.worldbank.org/v2/country/{code}/indicator/CM.MKT.LCAP.GD.ZS?format=json&per_page=1000"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            results.append({"Country": country, "WB_code": code, "Year": None, "MarketCap_pct_GDP": None, "Buffett_indicator": None, "Note": f"HTTP {r.status_code}"})
            continue
        payload = r.json()
        # payload[1] contiene una lista año-valor; buscamos el primer valor no nulo (más reciente)
        data_list = payload[1] if isinstance(payload, list) and len(payload) > 1 else []
        val = None
        year = None
        for entry in data_list:
            v = entry.get("value")
            if v is not None:
                val = float(v)
                year = int(entry.get("date")) if entry.get("date") is not None else None
                break
        if val is None:
            note = "Sin datos disponibles"
            results.append({"Country": country, "WB_code": code, "Year": None, "MarketCap_pct_GDP": None, "Buffett_indicator": None, "Note": note})
        else:
            buffett = val / 100.0  # market cap / GDP
            results.append({"Country": country, "WB_code": code, "Year": year, "MarketCap_pct_GDP": val, "Buffett_indicator": buffett, "Note": "OK"})
    except Exception as e:
        results.append({"Country": country, "WB_code": code, "Year": None, "MarketCap_pct_GDP": None, "Buffett_indicator": None, "Note": str(e)})

# DataFrame final
df_buffett = pd.DataFrame(results)
# Ordenar por indicador Buffett descendente (mayor market cap relativo al PIB)
df_buffett_sorted = df_buffett.sort_values(by="Buffett_indicator", ascending=False, na_position='last').reset_index(drop=True)

# Mostrar con formato
pd.options.display.float_format = '{:,.2f}'.format
print("Indicador Buffett (Market Cap / PIB) por país — fuente: World Bank (CM.MKT.LCAP.GD.ZS).\n")
display(df_buffett_sorted)

# Nota: Para países que no aparecen o tienen datos nulos, podés introducir manualmente la cifra
# de market cap o el GDP y calcular el indicador: indicator = market_cap / gdp.


Indicador Buffett (Market Cap / PIB) por país — fuente: World Bank (CM.MKT.LCAP.GD.ZS).



,Country,WB_code,Year,MarketCap_pct_GDP,Buffett_indicator,Note
0,Hong Kong,HK,"2,024.00","1,118.24",11.18,OK
1,Sudáfrica,ZA,"2,024.00",245.72,2.46,OK
2,Suiza,CH,"2,024.00",210.49,2.10,OK
3,Japón,JP,"2,024.00",156.69,1.57,OK
4,Canadá,CA,"2,024.00",150.40,1.50,OK
5,India,IN,"2,024.00",131.24,1.31,OK
6,Países Bajos,NL,"2,017.00",129.69,1.30,OK
7,Singapur,SG,"2,024.00",116.49,1.16,OK
8,Malasia,MY,"2,024.00",106.45,1.06,OK
9,Australia,AU,"2,024.00",98.87,0.99,OK


In [5]:
# Gráfico: evolución del Price-to-Earnings (P/E) para un ticker (por defecto MCHI)
# Intentos sucesivos de construir una serie de EPS histórica; si no hay datos, cae en trailingEps.

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display


def plot_pe_history(ticker='MCHI', period='5y', interval='1d', max_pe_clip=200):
    """Descarga precios e intenta construir una serie de P/E y la grafica.

    Estrategia:
    1. Descargar precios (Close) para el periodo solicitado.
    2. Intentar obtener EPS históricos:
       - Si hay datos trimestrales/annuales en yfinance, construir EPS por periodo y alinear con precios.
       - Si no hay, usar info['trailingEps'] como fallback (constante).
    3. Calcular P/E como precio / EPS (donde EPS>0), marcar zonas inválidas.
    4. Mostrar gráfico y tabla resumen (últimos valores).
    """
    t = yf.Ticker(ticker)

    # 1) Precios
    hist = t.history(period=period, interval=interval, auto_adjust=False)
    if hist is None or hist.empty:
        raise RuntimeError(f"No se pudieron descargar precios para {ticker} con periodo={period}")
    prices = hist['Close'].astype(float).copy()

    # 2) Intentos de construir EPS histórico
    eps_ts = None
    eps_source = None

    # A) Intentar con datos de 'earnings' (anual) y sharesOutstanding
    try:
        earnings = t.earnings if hasattr(t, 'earnings') else None
        info = t.info if hasattr(t, 'info') else {}
        shares = info.get('sharesOutstanding')
        if isinstance(earnings, pd.DataFrame) and not earnings.empty and shares is not None:
            # yfinance.earnings contiene una columna 'Earnings' por año (net income). No es EPS;
            # aproximamos EPS = Earnings / sharesOutstanding (uso aproximado con shares actuales)
            eps_idx = []
            eps_vals = []
            for yr in earnings.index:
                try:
                    val = float(earnings.loc[yr, 'Earnings'])
                    eps_idx.append(pd.Timestamp(f"{int(yr)}-12-31"))
                    eps_vals.append(val / float(shares))
                except Exception:
                    continue
            if eps_vals:
                eps_ts = pd.Series(data=eps_vals, index=pd.to_datetime(eps_idx)).sort_index()
                eps_source = 'earnings/sharesOutstanding (approx)'
    except Exception:
        pass

    # B) Intentar con quarterly financials -> Net Income / shares (más granular)
    if eps_ts is None:
        try:
            q_is = t.quarterly_financials if hasattr(t, 'quarterly_financials') else None
            if isinstance(q_is, pd.DataFrame) and not q_is.empty and 'Net Income' in q_is.index:
                # q_is columns son fechas; calcular EPS por columna usando info.sharesOutstanding si disponible
                if shares is not None:
                    eps_idx = []
                    eps_vals = []
                    for col in q_is.columns:
                        try:
                            ni = _as_float(q_is.loc['Net Income', col])
                            if np.isfinite(ni):
                                eps_idx.append(pd.to_datetime(col))
                                eps_vals.append(float(ni) / float(shares))
                        except Exception:
                            continue
                    if eps_vals:
                        eps_ts = pd.Series(data=eps_vals, index=pd.to_datetime(eps_idx)).sort_index()
                        eps_source = 'quarterly Net Income / sharesOutstanding (approx)'
        except Exception:
            pass

    # C) Intentar usar info['trailingEps'] como fallback (constante)
    if eps_ts is None:
        try:
            info = t.info if hasattr(t, 'info') else {}
            trailing_eps = info.get('trailingEps', None)
            if trailing_eps is not None and not (isinstance(trailing_eps, float) and not np.isfinite(trailing_eps)):
                trailing_eps = float(trailing_eps)
                # construir serie constante con el mismo índice que 'prices'
                eps_ts = pd.Series(trailing_eps, index=prices.index)
                eps_source = 'trailingEps (constant fallback)'
        except Exception:
            eps_ts = None
            eps_source = None

    if eps_ts is None or eps_ts.empty:
        raise RuntimeError(f"No fue posible construir una serie de EPS para {ticker}. Fuentes probadas: earnings, quarterly_financials, trailingEps.")

    # Si eps_ts es periódica (por años o trimestres), expandir a índice diario y forward-fill
    if not eps_ts.index.equals(prices.index):
        # reindex to price index using forward-fill/backfill as reasonable
        eps_daily = eps_ts.sort_index().reindex(prices.index, method='ffill')
        # si todavía hay NaNs al inicio, backfill
        eps_daily = eps_daily.fillna(method='bfill')
    else:
        eps_daily = eps_ts

    # Asegurar tipo float
    eps_daily = pd.to_numeric(eps_daily, errors='coerce')

    # 3) Calcular P/E (solo donde EPS > 0)
    pe = pd.Series(index=prices.index, dtype=float)
    valid = (eps_daily > 0) & (prices > 0)
    pe[valid] = prices[valid] / eps_daily[valid]
    # valores no válidos quedan como NaN

    # Clip para visualización
    pe_plot = pe.copy()
    if max_pe_clip is not None:
        pe_plot = pe_plot.clip(upper=max_pe_clip)

    # 4) Gráfico
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(pe_plot.index, pe_plot.values, label=f'P/E ({ticker})', color='tab:blue')
    ax.set_ylabel('P/E')
    ax.set_title(f'Evolución del P/E para {ticker} (periodo={period})\nFuente EPS: {eps_source}')
    ax.grid(axis='y', alpha=0.3)

    # Mostrar línea de mediana y último valor
    med = pe.dropna().median() if not pe.dropna().empty else np.nan
    last = pe.dropna().iloc[-1] if not pe.dropna().empty else np.nan
    if np.isfinite(med):
        ax.axhline(med, color='grey', linestyle='--', linewidth=1, label=f'Mediana P/E={med:.2f}')
    if np.isfinite(last):
        ax.annotate(f'Último P/E={last:.2f}', xy=(pe.dropna().index[-1], last), xytext=(-120, 10), textcoords='offset points', arrowprops=dict(arrowstyle='->', color='black'))

    ax.legend(loc='upper left')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

    # Tabla resumen
    summary = pd.DataFrame({
        'Último Precio': [prices.iloc[-1]],
        'Último EPS (aprox)': [eps_daily.dropna().iloc[-1] if not eps_daily.dropna().empty else np.nan],
        'Último P/E': [last]
    })
    display(summary)


# Uso: ejecutar esta celda y luego llamar plot_pe_history('MCHI', period='5y')
print('Celda lista. Llamá a plot_pe_history("MCHI", period="5y") para generar el gráfico.')

# Llamada automática para mostrar el gráfico inmediatamente al ejecutar la celda
try:
    plot_pe_history('MCHI', period='5y')
except Exception as _e:
    print(f"No se pudo generar el gráfico automáticamente: {_e}")

Celda lista. Llamá a plot_pe_history("MCHI", period="5y") para generar el gráfico.


c:\Users\nicon\anaconda3\Lib\site-packages\yfinance\scrapers\fundamentals.py:36: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


No se pudo generar el gráfico automáticamente: No fue posible construir una serie de EPS para MCHI. Fuentes probadas: earnings, quarterly_financials, trailingEps.
